In [8]:
with open("api_key.txt", "r") as f:
    API_KEY = f.read()

import sqlite3

db = sqlite3.connect("instance/db.sqlite", detect_types=sqlite3.PARSE_DECLTYPES)
db.row_factory = sqlite3.Row

headers = {"accept": "application/json", "Authorization": f"Bearer {API_KEY}"}

In [9]:
from time import sleep

import requests
from tqdm import tqdm

response = requests.get(
    "https://api.themoviedb.org/3/genre/movie/list",
    headers=headers,
    params={"language": "pt-BR"},
)

genres = response.json()["genres"]

In [10]:
genres_ids_mapping = {}

try:
    for genre_id in genres:
        cursor = db.execute(
            "INSERT INTO genero(descricao) VALUES (?)", (genre_id["name"],)
        )
        id_bd = cursor.lastrowid

        genres_ids_mapping[genre_id["id"]] = id_bd

    db.commit()

except Exception:
    db.rollback()
    raise

In [11]:
base_url_search_movie = "https://api.themoviedb.org/3/search/movie"
filmes_busca = ["Harry Potter", "Toy Story", "Star Wars", "Star Trek"]

saidas = []
filmes = []

for filme in tqdm(filmes_busca):

    params = {
        "query": filme,
        "include_adult": "false",
        "language": "pt-BR",
        "page": 1,
    }

    response = requests.get(base_url_search_movie, params=params, headers=headers)

    response_json = response.json()
    results = response_json["results"]
    for result in results:
        filmes.append(result)
    sleep(5)

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:20<00:00,  5.15s/it]


In [12]:
base_url = "https://api.themoviedb.org/3/movie/"

for filme in tqdm(filmes):
    url = base_url + str(filme["id"])
    details = requests.get(
        f"https://api.themoviedb.org/3/movie/{filme['id']}",
        headers=headers,
        params={"language": "pt-BR"},
    )
    filme["details"] = details.json()
    sleep(2)

100%|██████████| 80/80 [03:09<00:00,  2.37s/it]


In [13]:
try:
    for filme in filmes:
        titulo = filme["title"]
        titulo_original = filme["original_title"]
        data_lancamento = filme["release_date"]
        duracao = filme["details"]["runtime"]
        sinopse = filme["overview"]
        cursor = db.execute(
            """INSERT INTO filme(titulo, titulo_original, data_lancamento, duracao, sinopse) 
                   VALUES (?, ?, ?, ?, ?)""",
            (
                titulo,
                titulo_original,
                data_lancamento,
                duracao,
                sinopse,
            ),
        )

        filme_id = cursor.lastrowid
        for genre_id in filme["genre_ids"]:
            id_genero = genres_ids_mapping[genre_id]
            db.execute("INSERT INTO filme_genero VALUES (?, ?)", (filme_id, id_genero))

    db.commit()

except Exception:
    db.rollback()
    raise